# NB07 · Benchmark 迁移：评测协议对比学

| | |
|---|---|
| **目标** | 搞清「同一个 80% 在不同 benchmark 含金量差多少」——横向评测的方法论起点 |
| **前置** | NB03；`pip install metaworld`（或选 LIBERO） |
| **预计耗时** | 1 天 |
| **产出物** | `results/NB07.json`（协议对比表） |
| **通过标准** | 能量化说明两个 benchmark 的地板、判定松紧、初始分布宽窄 |

规则：从上到下顺序执行；每个 ✅ 检查点必须核对；最后的复盘必须填写并 commit。


In [ ]:
# Meta-World 最小接入（LIBERO 亦可，安装更重；二选一）
import random
import numpy as np
import metaworld

TASK = "pick-place-v2"
mt1 = metaworld.MT1(TASK, seed=0)
env = mt1.train_classes[TASK]()
env.set_task(random.choice(mt1.train_tasks))
obs, _ = env.reset()
print("obs dim:", obs.shape, " action space:", env.action_space)

In [ ]:
# 协议参数 dump：这就是"评测协议"的解剖
print("max_path_length:", env.max_path_length)
# 初始状态分布：reset 10 次看对象初始位置散布多大
inits = []
for _ in range(10):
    obs, _ = env.reset()
    inits.append(obs[:6].copy())
inits = np.array(inits)
print("初始状态前 6 维的 std:", inits.std(axis=0).round(4))
# success 判定：step 后 info["success"]——去 metaworld 源码里找到它的定义（距离阈值是多少？）

In [ ]:
# 地板测量：random policy 的 success——benchmark 的"零分线"不是 0
def random_rollouts(env, n=50):
    wins = 0
    for _ in range(n):
        env.set_task(random.choice(mt1.train_tasks))
        obs, _ = env.reset()
        for _ in range(env.max_path_length):
            obs, r, terminated, truncated, info = env.step(env.action_space.sample())
            if info.get("success", 0):
                wins += 1; break
            if terminated or truncated: break
    return wins / n

floor = random_rollouts(env)
print(f"random policy success = {floor:.1%}  ← 任何报告的数字先减掉这个地板再比较")

In [ ]:
import pandas as pd
import nbutils

comparison = pd.DataFrame({
    "pusht": {
        "success 判定": "T 块覆盖率 > 阈值（连续量取阈值）",
        "max steps": "300 (gym_pusht 默认，核实你的版本)",
        "初始分布": "T 块位姿随机（范围去源码核实）",
        "random 地板": "自己测：复用左边的方法",
        "随机性来源": "初始位姿 + (policy 采样?)",
    },
    "metaworld-" + TASK: {
        "success 判定": "物体到目标距离 < 阈值（去源码抄出数字）",
        "max steps": str(env.max_path_length),
        "初始分布": f"std={inits.std(axis=0).round(3).tolist()}",
        "random 地板": f"{floor:.1%}",
        "随机性来源": "task 采样 + 初始位姿",
    },
})
display(comparison)
nbutils.log_result("NB07", {"task": TASK, "random_floor": floor,
                            "comparison": comparison.to_dict()})

## 分析：含金量核算

1. 把表里的"核实"项全部替换成源码里抄出的真数字（附文件/行号）——**可复核是权威的最小单位**。
2. 写出换算句式：「pusht 上的 80% 大致相当于 metaworld pick-place 上的 __%，因为地板差 __、判定松紧差 __、初始分布宽窄差 __。」不要求精确，要求每一项有依据。
3. 推论：如果你来设计 observatory 的横向评测面板，两个来源的 success rate 能直接放在一列吗？需要哪些元数据字段才能对齐？——把字段列表写下来，这就是统一 schema 的雏形（冷启动 issue #1 的弹药）。


## 复盘（必填，不填不算完成这本 notebook）

> 复盘写在这里并 commit。允许粗糙，禁止事后美化。

- **预期 vs 实际**：
- **最大的一个意外**：
- **卡最久的一步和根因**：
- **用一句话向非技术人解释本次学到的东西**：
- **进入下一本之前要做的一个动作**：
